# RITINI: Inferring Dynamic Regulatory Interaction Graphs from Time Series Data with Perturbations

Prerequisites:
- Trained MIOFlow and decoded trajectories back to gene space.

In this notebook we will:
- Run RITINI to infer gene dynamics in gene regulatory networks

# Import libraries, set path and device

In [1]:
!git clone https://github.com/KrishnaswamyLab/omics_toolbox.git

fatal: destination path 'omics_toolbox' already exists and is not an empty directory.


In [2]:
!pip install numpy==1.26.4 torch==2.2.1 scanpy python-dotenv loguru torchdiffeq gdown torchdata==0.7.1 dgl

In [3]:
%cd /content/omics_toolbox
!pip install -e .

/content/omics_toolbox
Obtaining file:///content/omics_toolbox
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for omics_toolbox (pyproject.toml) ... done
  Created wheel for omics_toolbox: filename=omics_toolbox-0.0.1-py3-none-any.whl size=2327 sha256=ba9452385e61d91f78bd6a7dbe84729a5e557d991c2021288ea0af5c0132fee9
  Stored in directory: /tmp/pip-ephem-wheel-cache-h74kbdpe/wheels/27/48/91/55be6706e82e7e82444d88bae02977c8cb9d6d5005a7161f30
Successfully built omics_toolbox
  Attempting uninstall: omics_toolbox
    Found existing installation: omics_toolbox 0.0.1
    Uninstalling omics_toolbox-0.0.1:
      Successfully uninstalled omics_toolbox-0.0.1


In [4]:
# Standard library imports
import warnings
from sklearn.preprocessing import StandardScaler
import os
import pickle
import dgl
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scanpy as sc
import torch
# System and file handling libraries
import gdown               # Google Drive file downloader for accessing shared datasets
from torch.optim.lr_scheduler import StepLR
# Local application imports
from omics_toolbox.gode.utils import get_device
from omics_toolbox.gode.data import make_train_test_dataframe
from omics_toolbox.ritini_module import ritini

# Suppress specific warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=".*unique with argument that is not not a Series.*"
)

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


In [5]:
device = get_device()

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

seed = 3
torch.manual_seed(seed)
np.random.seed(seed)

# Load and Preprocess Dataset

Load the dataset (output from MIOFlow) of shape ((n_timepoints, n_trajectrories, n_genes))

In [6]:
traj_file_id = "18JNPlp3nPCVtJ9gTBYEbCJ98Evp-0jvg" #File id in google drive for our data
traj_url = f"https://drive.google.com/uc?id={traj_file_id}"
traj_output = "traj_data.pkl"
gdown.download(traj_url, traj_output, quiet=False)

genes_file_id = "1GrKIcZjfvIUL7qwunV_20Sw4T9KUNkyd" #File id in google drive for our data
genes_url = f"https://drive.google.com/uc?id={genes_file_id}"
genes_output = "genes_data.pkl"
gdown.download(genes_url, genes_output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=18JNPlp3nPCVtJ9gTBYEbCJ98Evp-0jvg
From (redirected): https://drive.google.com/uc?id=18JNPlp3nPCVtJ9gTBYEbCJ98Evp-0jvg&confirm=t&uuid=4223e036-7fd5-4527-b182-d4feebe61b1d
To: /content/omics_toolbox/traj_data.pkl
100%|██████████| 1.72G/1.72G [00:23<00:00, 72.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1GrKIcZjfvIUL7qwunV_20Sw4T9KUNkyd
To: /content/omics_toolbox/genes_data.pkl
100%|██████████| 203k/203k [00:00<00:00, 15.6MB/s]


'genes_data.pkl'

In [7]:
"""
Load MIOFlow inferred trajectories.
Shape (n_timepoints, n_trajectrories, n_genes)
"""
# trajectories = np.load(f"/vast/palmer/pi/krishnaswamy_smita/hcd22/nikjoshi_new_2/results/dimchanger/dimchanger_{subset_name}/{flow_name}_gene_sp.npy", allow_pickle=True)
# trajectories = np.load('../../results/scRNAseq/trajectories_gene_space.npy', allow_pickle=True)
with open('/content/omics_toolbox/traj_data.pkl', 'rb') as f:
    traj_data = pickle.load(f)
trajectories = traj_data
"""
Load gene names for plotting purposes
Shape (n_genes,)
"""
# genes = np.load(f'/vast/palmer/pi/krishnaswamy_smita/hcd22/nikjoshi_new_2/results/granger/granger_{subset_name}/{flow_name}_gene_names.npy', allow_pickle=True)
# adata = sc.read('../../data/processed/adata_mioflow.h5ad')
# genes = adata.var_names
with open('/content/omics_toolbox/genes_data.pkl', 'rb') as f:
    genes = pickle.load(f)
genes = pd.Series(genes).astype(str) \
        .str.replace(r'\s*\(ENSG[^\)]*\)', '', regex=True) \
        .to_numpy()
"""
Annotations are clusters of trajectories for a specific lineage. Here we assume just one set of trajectories.
All labeled as 0.
"""
annotations = np.zeros(trajectories.shape[0])

"""
For simplicity, we will just train on the mean trajectory.
"""
mean_trajectories = trajectories.mean(axis=1, keepdims=True)

traj_data = {
    'trajectories': mean_trajectories,
    'genes': genes,
    'annotations': annotations
}

In [ ]:
top_genes_file_id = "10HCDfG1C1-Oay6G483grKQqye_PEsYZH" #File id in google drive for our data
top_genes_url = f"https://drive.google.com/uc?id={top_genes_file_id}"
top_genes_output = "cancer_granger_prior_top_genes_20.pkl"
gdown.download(top_genes_url, top_genes_output, quiet=False)
with open('/content/omics_toolbox/cancer_granger_prior_top_genes_20.pkl', 'rb') as f:
    top_genes = pickle.load(f)

networkx_file_id = "1K8opErzOYwNJgZhLtoJV0UynlZ-pE8ix" #File id in google drive for our data
networkx_url = f"https://drive.google.com/uc?id={networkx_file_id}"
networkx_output = "cancer_granger_prior_graph_nx_20.pkl"
gdown.download(networkx_url, networkx_output, quiet=False)
# Load NetworkX graph G
with open('/content/omics_toolbox/cancer_granger_prior_graph_nx_20.pkl', 'rb') as f:
    G = pickle.load(f)

dgl_file_id = "1u6DLNfuLtLVZLhXgaPOrfIYxD5_GEmb3" #File id in google drive for our data
dgl_url = f"https://drive.google.com/uc?id={dgl_file_id}"
dgl_output = "cancer_granger_prior_graph_dgl_20.pkl"
gdown.download(dgl_url, dgl_output, quiet=False)
# Load DGL graph g
with open('/content/omics_toolbox/cancer_granger_prior_graph_dgl_20.pkl', 'rb') as f:
    g = pickle.load(f)

Downloading...
From: https://drive.google.com/uc?id=1lz9NSYEYcXICHrSHtFUqMelFpRJFgJqk
To: /content/omics_toolbox/cancer_granger_prior_top_genes_100.pkl
100%|██████████| 1.04k/1.04k [00:00<00:00, 2.44MB/s]
Downloading...
From: https://drive.google.com/uc?id=1cFJVpWoK8QcX-Q0C-GMZiKH1MuddK4KR
To: /content/omics_toolbox/cancer_granger_prior_graph_nx_100.pkl
100%|██████████| 9.19k/9.19k [00:00<00:00, 16.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=11KFV3YXYJ4hN7yDG5drRY2RTeVhND0YH
To: /content/omics_toolbox/cancer_granger_prior_graph_dgl_100.pkl
100%|██████████| 6.93k/6.93k [00:00<00:00, 12.5MB/s]


In [9]:
"""
Create the adjacency matrix.
"""
adjacency_matrix = nx.adjacency_matrix(G)
adjacency_matrix_negative = 1 - adjacency_matrix.todense() - np.eye(g.number_of_nodes())
edges = list(G.edges())
if len(edges) > 0:
    u, v = np.array(edges).T
    u = torch.tensor(u, dtype=torch.int32)
    v = torch.tensor(v, dtype=torch.int32)
else:
    u = v = torch.tensor([], dtype=torch.int32)

ref_g = g.to_networkx()
ref_pos = nx.spring_layout(ref_g.to_undirected(), seed=seed)
top_genes_for_plotting = top_genes[:10]
# Color and label
for idx, node in enumerate(ref_g.nodes()):
    ref_g.nodes[node]['color'] = plt.get_cmap('viridis', len(top_genes))(idx)
    ref_g.nodes[node]['label'] = top_genes[idx]

In [10]:
# """
# """
# # Mapping for edge ids
edge_ids = np.arange(g.number_of_edges())

# # Shuffle
edge_ids = np.random.permutation(edge_ids)

test_size_percent = 30
test_size_fraction = test_size_percent / 100

edge_test_size = int(len(edge_ids) * test_size_fraction)


In [11]:
top_genes = [gene.replace('_y', '') for gene in top_genes]

# Train RiTINI
- Saves plots of the ground truth dynamics vs predicted dynamics

In [12]:
seen = set()
gene_subset_indices = []
for i, k in enumerate(traj_data['genes']):
    if k in top_genes and k not in seen:
        seen.add(k)
        gene_subset_indices.append(i)
gene_subset_indices = np.array(gene_subset_indices)

cell_subset_indices = np.random.choice(traj_data['trajectories'].shape[1], traj_data['trajectories'].shape[1], replace=False)

In [13]:
traj_data['genes'].shape

(21465,)

In [14]:
trajs = traj_data['trajectories']
trajs = trajs[::3]
trajs = trajs[:, cell_subset_indices]
trajs = trajs[:, :, gene_subset_indices]
traj_f = trajs.reshape(-1, trajs.shape[2])

Here we construct pseudotime and cell type annotations for training the dynamic graph model RiTINI

In [15]:
pseudotimes = np.linspace(0, 1, trajs.shape[0])

In [16]:
annot_repeated = np.repeat(traj_data['annotations'][cell_subset_indices], trajs.shape[0])
pt_repeated = np.tile(pseudotimes, trajs.shape[1])
df = pd.DataFrame(traj_f, columns=top_genes, index=[f'cell_{i}' for i in range(traj_f.shape[0])])
df['pseudotime'] = pt_repeated

df['cell_types'] = [f'cell_type_{a}' for a in annot_repeated]
num_cell_types = len(df['cell_types'].unique())

In [17]:
df_train, df_test = make_train_test_dataframe(df)

In [18]:
n_cells_at_t = df['pseudotime'].value_counts()[0]

time_bins = np.sort(df.pseudotime.unique())
cell_types = np.sort(df.cell_types.unique())

t0, *_, tn = time_bins
time_tensor = torch.Tensor(time_bins)#.to(device)

in_feats = cell_types.size * n_cells_at_t
out_feats = cell_types.size * n_cells_at_t
df_train[top_genes] = StandardScaler().fit_transform(df_train[top_genes])


We now initialize the RITINI module and define the training hyperparameters, which can be adjusted based on experimental needs.

In [19]:
# Create RITINI instance first
graph_trainer = ritini.RITINI(g, in_feats, out_feats, device)
# Initialize the model
model = graph_trainer.model
device = 'cpu'
model = model.to(device)
# Call the train_test method on the instance
train_g = graph_trainer.train_test(
    edge_ids, edge_test_size
)

In [20]:
"""
Hyperparameters for training
"""
optimizer = torch.optim.AdamW(model.parameters(), lr=0.1, weight_decay=5e-4)
scheduler = StepLR(optimizer, step_size=350, gamma=0.1)
criterion = torch.nn.MSELoss()

steps = 100
verbose_step = 1

lambda_l1 = 10
add_n = 5
del_n = 5
link_step = 2
sample_size = 10

Load the model and train RITINI, which will generate plots of predicted vs. ground truth gene expression dynamics, saved in the Results/ folder.

In [21]:
graph_trainer.train_loop(
        model, optimizer, scheduler, criterion, top_genes,
        train_g, df_train, n_cells_at_t, time_bins, steps, link_step, add_n, del_n,
        verbose_step, num_cell_types, cell_types, ref_pos, ref_g, DATA_DIR='/content')

Dynamics Prediction loss: 0.27714046835899353
Dynamics Prediction loss: 0.27679741382598877
Dynamics Prediction loss: 0.27669191360473633
Dynamics Prediction loss: 0.2766408920288086
Dynamics Prediction loss: 0.27660971879959106
Dynamics Prediction loss: 0.2765827476978302
Dynamics Prediction loss: 0.2765650153160095
Dynamics Prediction loss: 0.27655354142189026
Dynamics Prediction loss: 0.2765420079231262
Dynamics Prediction loss: 0.27653568983078003
Dynamics Prediction loss: 0.2765243649482727
Dynamics Prediction loss: 0.27652519941329956
Dynamics Prediction loss: 0.2765128016471863
Dynamics Prediction loss: 0.27651914954185486
Dynamics Prediction loss: 0.2765093445777893
Dynamics Prediction loss: 0.2765124440193176
Dynamics Prediction loss: 0.27650386095046997
Dynamics Prediction loss: 0.27650687098503113
Dynamics Prediction loss: 0.27649930119514465
Dynamics Prediction loss: 0.2765020728111267
Dynamics Prediction loss: 0.2764952778816223
Dynamics Prediction loss: 0.2764977812767029

In [22]:
graph_trainer.__dir__()

['g',
 'in_feats',
 'out_feats',
 'device',
 'model',
 '__module__',
 '__init__',
 '_build_model',
 'train_test',
 'train_loop',
 'test_loop',
 '__dict__',
 '__weakref__',
 '__doc__',
 '__new__',
 '__repr__',
 '__hash__',
 '__str__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__getstate__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']